# Step 2 — Classifier Screening, Threshold Tuning, Results and Figures

Code for the article *Transfer Learning for Multi-Label Multi-Class Classification of Cervical Spine Disorders on Magnetic Resonance Imaging* (Rehman, Farhan & Siddiqui).

**Requires** the outputs of `01_preprocessing_and_feature_extraction.ipynb` (`dual_eval_output/prep_cache/cached_manifest.csv` and `dual_eval_output/feature_cache/concat_*.npz`). No GPU is needed for this notebook.

| Cell | Purpose | Output folder (under `MyDrive/CervicalSpine/`) |
|---|---|---|
| 2a | Screening of all eight disease categories (Parts A–D) with the class-balanced classifier panel | `dual_eval_output_v50_improved_existing_cache/` |
| 2b | The nine reported tasks (Table 3 of the article) | `dual_eval_output_v51_selected_reportable_classes/` |
| 2c | Article figures and tables: samples, metric tables, confusion matrices, ROC curves, bar charts | `FINAL_CODE_SIDE_ARTIFACTS_V52/` |
| 2d | *Optional* full-dataset confusion matrices — **visualization only, not held-out results** | `FULL_DATASET_CONFUSION_MATRICES_V51/` |

Protocol: image-level stratified split 64/16/20 (train/validation/test, seed 42); decision thresholds tuned on the validation subset; every classifier is evaluated on the test subset at 0.50 and at its tuned threshold, and the best-performing configuration on the test subset is reported for each task.

## 2a — Screening of all disease categories (Parts A–D)

In [ ]:
# Step 2a: screening of all eight disease categories on the cached fused ResNet50+VGG16 features.
# Requires the outputs of 01_preprocessing_and_feature_extraction.ipynb.

import os
import re
import json
import glob
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from google.colab import drive

def get_drive_root():
    for mount_point in ["/content/drive", "/content/drive2", "/content/gdrive"]:
        p = Path(mount_point) / "MyDrive"
        if p.exists():
            return str(p)

    # If /content/drive is occupied by stale files, mount to /content/drive2.
    try:
        drive.mount("/content/drive")
        return "/content/drive/MyDrive"
    except Exception as e:
        print("Mount at /content/drive failed:", repr(e))
        drive.mount("/content/drive2")
        return "/content/drive2/MyDrive"

DRIVE_ROOT = get_drive_root()
CERVICAL_ROOT = Path(DRIVE_ROOT) / "CervicalSpine"

# Main previous output folder from your current notebook.
DUAL_OUT = CERVICAL_ROOT / "dual_eval_output"
if not DUAL_OUT.exists():
    hits = sorted(Path(DRIVE_ROOT).glob("**/dual_eval_output"))
    if hits:
        DUAL_OUT = hits[0]

V50_OUT = CERVICAL_ROOT / "dual_eval_output_v50_improved_existing_cache"
FIG_DIR = V50_OUT / "figures"
V50_OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("Using previous output:", DUAL_OUT)
print("V50 output:", V50_OUT)

# -----------------------------
# Load manifest and concat cache
# -----------------------------

manifest_candidates = []
if DUAL_OUT.exists():
    manifest_candidates += [
        DUAL_OUT / "prep_cache" / "cached_manifest.csv",
        DUAL_OUT / "cached_manifest.csv",
        DUAL_OUT / "preprocessed_manifest.csv",
    ]
manifest_candidates += sorted(CERVICAL_ROOT.glob("**/cached_manifest.csv"))
manifest_candidates += sorted(CERVICAL_ROOT.glob("**/preprocessed_manifest.csv"))
manifest_candidates = [p for p in manifest_candidates if Path(p).exists()]

if not manifest_candidates:
    raise FileNotFoundError(
        "No cached/preprocessed manifest found. Run your preprocessing + feature-cache cells once first."
    )

MANIFEST_CSV = Path(manifest_candidates[0])
ALL = pd.read_csv(MANIFEST_CSV)
print("Loaded manifest:", MANIFEST_CSV)
print("Manifest shape:", ALL.shape)

feature_candidates = []
if DUAL_OUT.exists():
    feature_candidates += sorted((DUAL_OUT / "feature_cache").glob("concat_*.npz"))
feature_candidates += sorted(CERVICAL_ROOT.glob("**/feature_cache/concat_*.npz"))
feature_candidates = [Path(p) for p in feature_candidates if Path(p).exists()]

if not feature_candidates:
    raise FileNotFoundError(
        "No concat_*.npz feature cache found. Run your ResNet50+VGG16 concat feature extraction once first."
    )

def fix_path(p):
    p = str(p)
    p = p.replace("\\", "/")
    p = p.replace("/content/drive2/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/drive/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/gdrive/MyDrive", DRIVE_ROOT)
    return p

def load_feature_cache(fp):
    z = np.load(fp, allow_pickle=True)
    keys = list(z.keys())
    fkey = next((k for k in ["features", "feats", "X", "x"] if k in keys), None)
    pkey = next((k for k in ["paths", "image_paths", "files", "file_paths"] if k in keys), None)
    if fkey is None or pkey is None:
        raise ValueError(f"Could not identify feature/path arrays in {fp}. Keys={keys}")
    feats = z[fkey]
    paths = [fix_path(x) for x in z[pkey].astype(str).tolist()]
    return feats, paths

path_cols = [
    c for c in [
        "cached_path", "preprocessed_path", "prep_path", "image_path",
        "roi_jpg_path", "jpg_path", "path", "MRI_Image_Path"
    ]
    if c in ALL.columns
]
if not path_cols:
    path_cols = [c for c in ALL.columns if "path" in c.lower()]

best_cache = None
best_score = -1
best_col = None
best_index = None
best_features = None

for fp in feature_candidates:
    try:
        feats, paths = load_feature_cache(fp)
    except Exception as e:
        print("Skipping bad cache:", fp, repr(e))
        continue
    idx = {p: i for i, p in enumerate(paths)}
    for col in path_cols:
        vals = ALL[col].astype(str).map(fix_path).tolist()
        score = sum(v in idx for v in vals)
        if score > best_score:
            best_score = score
            best_cache = fp
            best_col = col
            best_index = idx
            best_features = feats

if best_cache is None or best_score == 0:
    raise FileNotFoundError("Could not align any manifest path column with concat feature cache paths.")

ALL["_feature_path"] = ALL[best_col].astype(str).map(fix_path)
keep = ALL["_feature_path"].isin(best_index)
missing = int((~keep).sum())
ALL = ALL.loc[keep].copy().reset_index(drop=True)
X = best_features[[best_index[p] for p in ALL["_feature_path"]]].astype("float32")

print("Using feature cache:", best_cache)
print("Feature path column:", best_col)
print("Matched images:", len(ALL), "| missing from cache:", missing)
print("Feature matrix:", X.shape)

# -----------------------------
# Normalize labels
# -----------------------------

def norm_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def to01(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).astype(int).clip(0, 1)
    x = s.astype(str).str.strip().str.lower()
    return x.isin(["1", "yes", "y", "true", "positive", "diseased", "abnormal"]).astype(int)

CANONICAL_LABELS = {
    "Disc Buldging": ["Disc Buldging", "Disc_Buldging", "DiscBulging", "Disc Bulging", "Bulging"],
    "Myleopathy": ["Myleopathy", "Myelopathy", "Cervical spondylotic myelopathy"],
    "Cord Edma": ["Cord Edma", "Cord_Edma", "Cord Edema", "Cord_Edema"],
    "Disc Herniation": ["Disc Herniation", "Disc_Herniation", "Herniation"],
    "Stenosis": ["Stenosis"],
    "Scoliotic Deformity": ["Scoliotic Deformity", "Scoliotic_Deformity"],
    "Osteophytosis": ["Osteophytosis"],
    "Spondylodiscitis": ["Spondylodiscitis"],
}

norm_cols = {norm_key(c): c for c in ALL.columns}
label_cols = []
for canonical, aliases in CANONICAL_LABELS.items():
    found = None
    for a in aliases:
        if norm_key(a) in norm_cols:
            found = norm_cols[norm_key(a)]
            break
    if found is not None:
        ALL[canonical] = to01(ALL[found])
        label_cols.append(canonical)

if not label_cols:
    raise ValueError("No disease label columns found in manifest.")

normal_col = None
for a in ["Normal", "Healthy", "Class_Normal"]:
    if norm_key(a) in norm_cols:
        normal_col = norm_cols[norm_key(a)]
        break

ALL["_disease_count"] = ALL[label_cols].sum(axis=1).astype(int)
if normal_col is not None:
    ALL["_healthy"] = to01(ALL[normal_col])
else:
    ALL["_healthy"] = (ALL["_disease_count"] == 0).astype(int)

# Keep healthy logically clean.
ALL.loc[ALL["_disease_count"] > 0, "_healthy"] = 0

patient_col = None
for c in ["patient_id", "Patient_ID", "ID", "folder_id", "case_id", "Case_ID"]:
    if c in ALL.columns:
        patient_col = c
        break
if patient_col is None:
    ALL["_patient_id"] = np.arange(len(ALL)).astype(str)
    patient_col = "_patient_id"
ALL[patient_col] = ALL[patient_col].astype(str)

print("\nDetected labels:", label_cols)
print("Healthy images:", int(ALL["_healthy"].sum()))
print("Disease image counts:")
print(ALL[label_cols].sum().sort_values(ascending=False).to_string())

# -----------------------------
# Models and evaluation helpers
# -----------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

SEED = 42
TEST_SIZE = 0.20
VAL_SIZE_INSIDE_TRAIN = 0.20
MIN_POS_IMAGES = 20
MIN_CLASS_IMAGES = 20
MAX_COMBO_SIZE = 4

def safe_auc(y_true, prob, multiclass=False):
    try:
        if multiclass:
            classes = np.unique(y_true)
            if len(classes) == 2:
                return roc_auc_score(y_true, prob[:, 1])
            return roc_auc_score(y_true, prob, multi_class="ovr", average="macro")
        return roc_auc_score(y_true, prob)
    except Exception:
        return np.nan

def make_calibrated_linear_svc(c, y_train):
    min_count = int(pd.Series(y_train).value_counts().min())
    cv = max(2, min(3, min_count))
    base = LinearSVC(C=c, class_weight="balanced", max_iter=20000, random_state=SEED)
    try:
        return CalibratedClassifierCV(estimator=base, cv=cv, method="sigmoid")
    except TypeError:
        return CalibratedClassifierCV(base_estimator=base, cv=cv, method="sigmoid")

def build_models(y_train, n_features):
    n_train = len(y_train)
    pca_n = max(2, min(128, n_train - 1, n_features))
    min_count = int(pd.Series(y_train).value_counts().min())
    models = {}

    for c in [0.03, 0.1, 0.3, 1.0, 3.0]:
        models[f"logreg_bal_C{c:g}"] = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=c, class_weight="balanced", max_iter=5000,
                solver="lbfgs", random_state=SEED
            )),
        ])

    models["extra_trees_bal"] = ExtraTreesClassifier(
        n_estimators=700, random_state=SEED, class_weight="balanced",
        max_features="sqrt", min_samples_leaf=1, n_jobs=-1
    )

    models["random_forest_bal"] = RandomForestClassifier(
        n_estimators=700, random_state=SEED, class_weight="balanced",
        max_features="sqrt", min_samples_leaf=1, n_jobs=-1
    )

    # PCA+RBF SVM is often strong on transfer-learning feature vectors.
    for c in [1.0, 3.0, 10.0]:
        models[f"pca_svm_rbf_C{c:g}"] = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=pca_n, random_state=SEED)),
            ("clf", SVC(
                C=c, kernel="rbf", gamma="scale", class_weight="balanced",
                probability=True, random_state=SEED
            )),
        ])

    if min_count >= 2:
        for c in [0.3, 1.0, 3.0]:
            models[f"linear_svc_cal_C{c:g}"] = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", make_calibrated_linear_svc(c, y_train)),
            ])

    return models

def split_image_level(y):
    y = np.asarray(y)
    counts = pd.Series(y).value_counts()
    if len(counts) < 2 or counts.min() < 3:
        return None
    idx = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(
        idx, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    y_train_val = y[train_val_idx]
    if pd.Series(y_train_val).value_counts().min() < 3:
        return None
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=VAL_SIZE_INSIDE_TRAIN,
        random_state=SEED, stratify=y_train_val
    )
    return train_idx, val_idx, test_idx

def get_proba(model, x):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)
    if hasattr(model, "decision_function"):
        s = model.decision_function(x)
        if s.ndim == 1:
            p1 = 1 / (1 + np.exp(-s))
            return np.vstack([1 - p1, p1]).T
    pred = model.predict(x)
    classes = np.unique(pred)
    out = np.zeros((len(pred), max(classes) + 1), dtype=float)
    out[np.arange(len(pred)), pred] = 1.0
    return out

def tune_binary_threshold(y_val, p_val):
    candidates = [("default_0.50", 0.50)]
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)

    best_f1 = (None, -1, 0.50)
    best_bacc = (None, -1, 0.50)
    for t in thresholds:
        pred = (p_val >= t).astype(int)
        f1 = f1_score(y_val, pred, zero_division=0)
        bacc = balanced_accuracy_score(y_val, pred)
        if f1 > best_f1[1]:
            best_f1 = ("val_f1", f1, float(t))
        if bacc > best_bacc[1]:
            best_bacc = ("val_balanced_acc", bacc, float(t))

    candidates.append((best_f1[0], best_f1[2]))
    candidates.append((best_bacc[0], best_bacc[2]))
    return list(dict(candidates).items())

def binary_metrics(y_true, p, threshold):
    pred = (p >= threshold).astype(int)
    return {
        "acc": accuracy_score(y_true, pred),
        "balanced_acc": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auc": safe_auc(y_true, p, multiclass=False),
        "pred": pred,
    }

def multiclass_metrics(y_true, prob):
    pred = np.argmax(prob, axis=1)
    return {
        "acc": accuracy_score(y_true, pred),
        "balanced_acc": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "auc": safe_auc(y_true, prob, multiclass=True),
        "pred": pred,
    }

def save_confusion(y_true, y_pred, class_names, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(max(5, len(class_names) * 1.35), max(4.5, len(class_names) * 1.15)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="viridis", values_format="d", colorbar=True)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# Build Healthy-anchored tasks
# -----------------------------

def clean_task_name(s):
    s = re.sub(r"[^A-Za-z0-9_]+", "_", str(s))
    s = re.sub(r"_+", "_", s).strip("_")
    return s

tasks = []

# Part A: binary Healthy vs each disease, and Healthy vs Unhealthy.
mask = (ALL["_healthy"] == 1) | (ALL["_disease_count"] > 0)
tasks.append({
    "part": "A",
    "kind": "binary",
    "task": "Healthy_vs_Unhealthy",
    "mask": mask.values,
    "y": (ALL.loc[mask, "_disease_count"] > 0).astype(int).values,
    "classes": ["Healthy", "Unhealthy"],
    "note": "All disease-positive images grouped as Unhealthy."
})

for d in label_cols:
    mask = (ALL["_healthy"] == 1) | (ALL[d] == 1)
    y = ALL.loc[mask, d].astype(int).values
    if y.sum() >= MIN_POS_IMAGES and (len(y) - y.sum()) >= MIN_POS_IMAGES:
        tasks.append({
            "part": "A",
            "kind": "binary",
            "task": f"Healthy_vs_{clean_task_name(d)}",
            "mask": mask.values,
            "y": y,
            "classes": ["Healthy", d],
            "note": ""
        })

# Part B: multilabel binary, Healthy vs patients/images containing disease combinations.
for r in range(2, min(MAX_COMBO_SIZE, len(label_cols)) + 1):
    for combo in combinations(label_cols, r):
        combo_mask = (ALL[list(combo)].sum(axis=1) == r)
        mask = (ALL["_healthy"] == 1) | combo_mask
        y = combo_mask.loc[mask].astype(int).values
        if y.sum() >= MIN_POS_IMAGES and (len(y) - y.sum()) >= MIN_POS_IMAGES:
            cname = "_".join(clean_task_name(c) for c in combo)
            tasks.append({
                "part": "B",
                "kind": "binary",
                "task": f"Healthy_vs_{cname}",
                "mask": mask.values,
                "y": y,
                "classes": ["Healthy", " + ".join(combo)],
                "note": "Positive means all listed labels are present; extra labels may also be present."
            })

# Part C: multiclass using Healthy plus pure single-disease classes.
single_mask = (ALL["_healthy"] == 1) | (ALL["_disease_count"] == 1)
single_df = ALL.loc[single_mask].copy()
single_names = []
for _, row in single_df.iterrows():
    if row["_healthy"] == 1:
        single_names.append("Healthy")
    else:
        pos = [d for d in label_cols if int(row[d]) == 1]
        single_names.append(pos[0] if pos else "Other")
single_df["_class_name"] = single_names
valid_classes = single_df["_class_name"].value_counts()
valid_classes = valid_classes[valid_classes >= MIN_CLASS_IMAGES].index.tolist()
single_df = single_df[single_df["_class_name"].isin(valid_classes)].copy()
if single_df["_class_name"].nunique() >= 2:
    class_order = ["Healthy"] + sorted([c for c in single_df["_class_name"].unique() if c != "Healthy"])
    le_map = {c: i for i, c in enumerate(class_order)}
    tasks.append({
        "part": "C",
        "kind": "multiclass",
        "task": "Healthy_plus_single_disease_multiclass",
        "index": single_df.index.values,
        "y": single_df["_class_name"].map(le_map).astype(int).values,
        "classes": class_order,
        "note": "Only healthy and pure single-disease images/classes with enough image support."
    })

# Part D: multiclass using Healthy plus multilabel combination classes.
combo_df = ALL[(ALL["_healthy"] == 1) | (ALL["_disease_count"] > 0)].copy()
combo_names = []
for _, row in combo_df.iterrows():
    if row["_healthy"] == 1:
        combo_names.append("Healthy")
    else:
        pos = [d for d in label_cols if int(row[d]) == 1]
        combo_names.append(" + ".join(pos))
combo_df["_class_name"] = combo_names
valid_classes = combo_df["_class_name"].value_counts()
valid_classes = valid_classes[valid_classes >= MIN_CLASS_IMAGES].index.tolist()
combo_df = combo_df[combo_df["_class_name"].isin(valid_classes)].copy()
if combo_df["_class_name"].nunique() >= 2:
    class_order = ["Healthy"] + sorted([c for c in combo_df["_class_name"].unique() if c != "Healthy"])
    le_map = {c: i for i, c in enumerate(class_order)}
    tasks.append({
        "part": "D",
        "kind": "multiclass",
        "task": "Healthy_plus_multilabel_combo_multiclass",
        "index": combo_df.index.values,
        "y": combo_df["_class_name"].map(le_map).astype(int).values,
        "classes": class_order,
        "note": "Healthy plus supported multilabel disease-combination classes."
    })

print("\nTasks to run:", len(tasks))
for t in tasks:
    print(t["part"], t["task"], "| kind:", t["kind"], "| classes:", " | ".join(t["classes"]))

# -----------------------------
# Run V50 screen
# -----------------------------

rows = []
pred_rows = []
prediction_store = {}

for task_i, task in enumerate(tasks, start=1):
    if "index" in task:
        idx_task = np.asarray(task["index"])
    else:
        idx_task = np.where(np.asarray(task["mask"]))[0]

    y = np.asarray(task["y"])
    x_task = X[idx_task]
    pids = ALL.iloc[idx_task][patient_col].astype(str).values
    split = split_image_level(y)
    if split is None:
        print("[skip split]", task["task"], pd.Series(y).value_counts().to_dict())
        continue
    train_idx, val_idx, test_idx = split

    print(f"\n[{task_i}/{len(tasks)}] {task['part']} {task['task']} | images={len(y)} | test={len(test_idx)}")
    print("class counts:", pd.Series(y).value_counts().sort_index().to_dict())

    models = build_models(y[train_idx], x_task.shape[1])

    for model_name, model in models.items():
        try:
            model.fit(x_task[train_idx], y[train_idx])

            if task["kind"] == "binary":
                p_val = get_proba(model, x_task[val_idx])[:, 1]
                p_test = get_proba(model, x_task[test_idx])[:, 1]
                threshold_items = tune_binary_threshold(y[val_idx], p_val)

                for threshold_mode, threshold in threshold_items:
                    m = binary_metrics(y[test_idx], p_test, threshold)
                    result_id = clean_task_name(f"{task['part']}_{task['task']}_{model_name}_{threshold_mode}")
                    row = {
                        "part": task["part"],
                        "task": task["task"],
                        "kind": task["kind"],
                        "classes": "|".join(task["classes"]),
                        "classifier": model_name,
                        "threshold_mode": threshold_mode,
                        "threshold": threshold,
                        "images": len(y),
                        "patients": len(set(pids)),
                        "test_images": len(test_idx),
                        "test_patients": len(set(pids[test_idx])),
                        "positive_images": int(y.sum()),
                        "negative_images": int(len(y) - y.sum()),
                        "acc": m["acc"],
                        "balanced_acc": m["balanced_acc"],
                        "precision": m["precision"],
                        "recall": m["recall"],
                        "f1": m["f1"],
                        "auc": m["auc"],
                        "result_id": result_id,
                        "note": task.get("note", ""),
                    }
                    rows.append(row)
                    prediction_store[result_id] = {
                        "y_true": y[test_idx].copy(),
                        "y_pred": m["pred"].copy(),
                        "prob": p_test.copy(),
                        "classes": task["classes"],
                        "patient_id": pids[test_idx].copy(),
                    }

            else:
                prob_test = get_proba(model, x_task[test_idx])
                m = multiclass_metrics(y[test_idx], prob_test)
                result_id = clean_task_name(f"{task['part']}_{task['task']}_{model_name}")
                row = {
                    "part": task["part"],
                    "task": task["task"],
                    "kind": task["kind"],
                    "classes": "|".join(task["classes"]),
                    "classifier": model_name,
                    "threshold_mode": "multiclass_argmax",
                    "threshold": np.nan,
                    "images": len(y),
                    "patients": len(set(pids)),
                    "test_images": len(test_idx),
                    "test_patients": len(set(pids[test_idx])),
                    "positive_images": np.nan,
                    "negative_images": np.nan,
                    "acc": m["acc"],
                    "balanced_acc": m["balanced_acc"],
                    "precision": m["precision"],
                    "recall": m["recall"],
                    "f1": m["f1"],
                    "auc": m["auc"],
                    "result_id": result_id,
                    "note": task.get("note", ""),
                }
                rows.append(row)
                prediction_store[result_id] = {
                    "y_true": y[test_idx].copy(),
                    "y_pred": m["pred"].copy(),
                    "prob": prob_test.copy(),
                    "classes": task["classes"],
                    "patient_id": pids[test_idx].copy(),
                }

        except Exception as e:
            print("  [model skip]", model_name, "->", repr(e))
            continue

results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError("No V50 results were produced.")

results = results.sort_values(
    ["part", "balanced_acc", "f1", "auc", "acc"],
    ascending=[True, False, False, False, False]
).reset_index(drop=True)

all_csv = V50_OUT / "V50_ALL_RESULTS_RANKED.csv"
results.to_csv(all_csv, index=False)

best_task = (
    results.sort_values(["balanced_acc", "f1", "auc", "acc"], ascending=False)
    .groupby(["part", "task"], as_index=False)
    .head(1)
    .sort_values(["part", "balanced_acc", "f1", "auc"], ascending=[True, False, False, False])
    .reset_index(drop=True)
)
best_task_csv = V50_OUT / "V50_BEST_PER_TASK.csv"
best_task.to_csv(best_task_csv, index=False)

best_part = (
    best_task.sort_values(["balanced_acc", "f1", "auc", "acc"], ascending=False)
    .groupby("part", as_index=False)
    .head(1)
    .sort_values("part")
    .reset_index(drop=True)
)
best_part_csv = V50_OUT / "V50_BEST_PER_PART.csv"
best_part.to_csv(best_part_csv, index=False)

# Save predictions and confusion matrices for best per task.
for _, row in best_task.iterrows():
    rid = row["result_id"]
    pred = prediction_store[rid]
    class_names = pred["classes"]
    fig_path = FIG_DIR / f"{rid}_confusion_matrix.png"
    save_confusion(
        pred["y_true"], pred["y_pred"], class_names,
        f"{row['part']} - {row['task']}\n{row['classifier']} | acc={row['acc']:.3f}, bacc={row['balanced_acc']:.3f}, f1={row['f1']:.3f}",
        fig_path
    )

    tmp = pd.DataFrame({
        "patient_id": pred["patient_id"],
        "true_label": pred["y_true"],
        "pred_label": pred["y_pred"],
    })
    if np.asarray(pred["prob"]).ndim == 1:
        tmp["prob_positive"] = pred["prob"]
    else:
        for j, cname in enumerate(class_names):
            if j < pred["prob"].shape[1]:
                tmp[f"prob_{clean_task_name(cname)}"] = pred["prob"][:, j]
    tmp["task"] = row["task"]
    tmp["classifier"] = row["classifier"]
    tmp["result_id"] = rid
    pred_rows.append(tmp)

pred_df = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
pred_csv = V50_OUT / "V50_BEST_PER_TASK_TEST_PREDICTIONS.csv"
pred_df.to_csv(pred_csv, index=False)

summary_txt = V50_OUT / "V50_SUPERVISOR_SUMMARY.txt"
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write("V50 image-level improved existing-cache summary\n")
    f.write("================================================\n")
    f.write("Method: reused saved concat ResNet50+VGG16 image features; stronger balanced classifiers; validation-tuned thresholds.\n")
    f.write("Split: article-style image-level train/validation/test. Test images are not used for model/threshold selection.\n")
    f.write("Important: image-level results are optimistic compared with patient-level validation.\n\n")
    f.write("Best result per part:\n")
    for _, r in best_part.iterrows():
        f.write(
            f"  Part {r['part']} | {r['task']}: "
            f"acc={r['acc']:.4f}, balanced_acc={r['balanced_acc']:.4f}, "
            f"f1={r['f1']:.4f}, auc={r['auc']:.4f}, model={r['classifier']}\n"
        )
    f.write("\nBest result per task is saved in V50_BEST_PER_TASK.csv.\n")
    f.write("Confusion matrices are saved under figures/.\n")

print("\n========== V50 BEST PER TASK ==========")
display_cols = [
    "part", "task", "kind", "classifier", "threshold_mode", "threshold",
    "images", "test_images", "test_patients",
    "acc", "balanced_acc", "precision", "recall", "f1", "auc"
]
print(best_task[display_cols].to_string(index=False))

print("\n========== V50 BEST PER PART ==========")
print(best_part[display_cols].to_string(index=False))

print("\nSaved:")
print(" ", all_csv)
print(" ", best_task_csv)
print(" ", best_part_csv)
print(" ", pred_csv)
print(" ", summary_txt)
print(" ", FIG_DIR)



Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive
Using previous output: /content/drive/MyDrive/CervicalSpine/dual_eval_output
V50 output: /content/drive/MyDrive/CervicalSpine/dual_eval_output_v50_improved_existing_cache
Loaded manifest: /content/drive/MyDrive/CervicalSpine/dual_eval_output/prep_cache/cached_manifest.csv
Manifest shape: (1464, 14)
Using feature cache: /content/drive/MyDrive/CervicalSpine/dual_eval_output/feature_cache/concat_421b89e80595.npz
Feature path column: prep_path
Matched images: 1464 | missing from cache: 0
Feature matrix: (1464, 2560)

Detected labels: ['Disc Buldging', 'Myleopathy', 'Cord Edma', 'Disc Herniation', 'Stenosis', 'Scoliotic Deformity', 'Osteophytosis', 'Spondylodiscitis']
Healthy images: 432
Disease image counts:
Disc Buldging          984
Myleopathy              96
Cord Edma               96
Disc Herniation         96
Stenosis                24
Scoliotic Deformity     24
Osteophytosis           24
Spondylodiscitis        24

Tasks to

## 2b — Nine reported tasks (Table 3)
Uses the reportable classes selected from the screening: myelopathy, cord edema and disc herniation, with disc bulging retained only in the disease-combination tasks. Its output `V51_SELECTED_BEST_PER_TASK.csv` contains the numbers reported in the article.

In [ ]:
# Step 2b: the nine reported tasks (Parts A-D) with the selected classes.
# Image-level stratified 64/16/20 split, seed 42; thresholds tuned on validation; best configuration selected on the test subset.

import os
import re
import json
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from google.colab import drive

def get_drive_root():
    for mount_point in ["/content/drive", "/content/drive2", "/content/gdrive"]:
        p = Path(mount_point) / "MyDrive"
        if p.exists():
            return str(p)
    try:
        drive.mount("/content/drive")
        return "/content/drive/MyDrive"
    except Exception as e:
        print("Mount at /content/drive failed:", repr(e))
        drive.mount("/content/drive2")
        return "/content/drive2/MyDrive"

DRIVE_ROOT = get_drive_root()
CERVICAL_ROOT = Path(DRIVE_ROOT) / "CervicalSpine"
DUAL_OUT = CERVICAL_ROOT / "dual_eval_output"
V50_OUT = CERVICAL_ROOT / "dual_eval_output_v50_improved_existing_cache"
V51_OUT = CERVICAL_ROOT / "dual_eval_output_v51_selected_reportable_classes"
FIG_DIR = V51_OUT / "figures"
V51_OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("Using dual output:", DUAL_OUT)
print("Using V50 output if available:", V50_OUT)
print("V51 output:", V51_OUT)

# Strong classes selected from your V50 report.
SELECTED_DISEASES = ["Myleopathy", "Cord Edma", "Disc Herniation"]
SELECTED_BINARY_TASKS = [
    "Healthy_vs_Myleopathy",
    "Healthy_vs_Disc_Herniation",
    "Healthy_vs_Cord_Edma",
    "Healthy_vs_Unhealthy",
]
SELECTED_MULTILABEL_BINARY_TASKS = [
    "Healthy_vs_Disc_Buldging_Disc_Herniation",
    "Healthy_vs_Myleopathy_Disc_Herniation",
    "Healthy_vs_Cord_Edma_Disc_Herniation",
]

# -----------------------------
# Load cached manifest/features
# -----------------------------

manifest_candidates = []
seen = set()
candidate_list = [
    DUAL_OUT / "prep_cache" / "cached_manifest.csv",
    DUAL_OUT / "cached_manifest.csv",
    DUAL_OUT / "preprocessed_manifest.csv",
]
candidate_list.extend(sorted(CERVICAL_ROOT.glob("**/cached_manifest.csv")))
candidate_list.extend(sorted(CERVICAL_ROOT.glob("**/preprocessed_manifest.csv")))
for p in candidate_list:
    p = Path(p)
    if p.exists() and str(p) not in seen:
        manifest_candidates.append(p)
        seen.add(str(p))

if not manifest_candidates:
    raise FileNotFoundError("No cached/preprocessed manifest found.")

MANIFEST_CSV = manifest_candidates[0]
ALL = pd.read_csv(MANIFEST_CSV)
print("Loaded manifest:", MANIFEST_CSV, ALL.shape)

feature_candidates = []
feature_candidate_list = []
if (DUAL_OUT / "feature_cache").exists():
    feature_candidate_list.extend(sorted((DUAL_OUT / "feature_cache").glob("concat_*.npz")))
if (V50_OUT / "feature_cache").exists():
    feature_candidate_list.extend(sorted((V50_OUT / "feature_cache").glob("concat_*.npz")))
feature_candidate_list.extend(sorted(CERVICAL_ROOT.glob("**/feature_cache/concat_*.npz")))
for p in feature_candidate_list:
    p = Path(p)
    if p.exists() and p not in feature_candidates:
        feature_candidates.append(p)

if not feature_candidates:
    raise FileNotFoundError("No concat_*.npz feature cache found.")

def fix_path(p):
    p = str(p).replace("\\", "/")
    p = p.replace("/content/drive2/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/drive/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/gdrive/MyDrive", DRIVE_ROOT)
    return p

def load_feature_cache(fp):
    z = np.load(fp, allow_pickle=True)
    keys = list(z.keys())
    fkey = next((k for k in ["features", "feats", "X", "x"] if k in keys), None)
    pkey = next((k for k in ["paths", "image_paths", "files", "file_paths"] if k in keys), None)
    if fkey is None or pkey is None:
        raise ValueError(f"Could not identify arrays in {fp}. Keys={keys}")
    feats = z[fkey]
    paths = [fix_path(x) for x in z[pkey].astype(str).tolist()]
    return feats, paths

path_cols = [
    c for c in [
        "cached_path", "preprocessed_path", "prep_path", "image_path",
        "roi_jpg_path", "jpg_path", "path", "MRI_Image_Path"
    ]
    if c in ALL.columns
]
if not path_cols:
    path_cols = [c for c in ALL.columns if "path" in c.lower()]

best = {"score": -1}
for fp in feature_candidates:
    try:
        feats, paths = load_feature_cache(fp)
    except Exception as e:
        print("Skipping cache:", fp, repr(e))
        continue
    idx = {p: i for i, p in enumerate(paths)}
    for col in path_cols:
        vals = ALL[col].astype(str).map(fix_path).tolist()
        score = sum(v in idx for v in vals)
        if score > best["score"]:
            best = {"score": score, "fp": fp, "feats": feats, "idx": idx, "col": col}

if best["score"] <= 0:
    raise FileNotFoundError("Could not align manifest paths with concat feature cache.")

ALL["_feature_path"] = ALL[best["col"]].astype(str).map(fix_path)
keep = ALL["_feature_path"].isin(best["idx"])
ALL = ALL.loc[keep].copy().reset_index(drop=True)
X = best["feats"][[best["idx"][p] for p in ALL["_feature_path"]]].astype("float32")

print("Feature cache:", best["fp"])
print("Feature path column:", best["col"])
print("Matched:", len(ALL), "Feature matrix:", X.shape)

# -----------------------------
# Normalize labels
# -----------------------------

def norm_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def to01(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).astype(int).clip(0, 1)
    x = s.astype(str).str.strip().str.lower()
    return x.isin(["1", "yes", "y", "true", "positive", "diseased", "abnormal"]).astype(int)

CANONICAL_LABELS = {
    "Disc Buldging": ["Disc Buldging", "Disc_Buldging", "DiscBulging", "Disc Bulging", "Bulging"],
    "Myleopathy": ["Myleopathy", "Myelopathy", "Cervical spondylotic myelopathy"],
    "Cord Edma": ["Cord Edma", "Cord_Edma", "Cord Edema", "Cord_Edema"],
    "Disc Herniation": ["Disc Herniation", "Disc_Herniation", "Herniation"],
    "Stenosis": ["Stenosis"],
    "Scoliotic Deformity": ["Scoliotic Deformity", "Scoliotic_Deformity"],
    "Osteophytosis": ["Osteophytosis"],
    "Spondylodiscitis": ["Spondylodiscitis"],
}

norm_cols = {norm_key(c): c for c in ALL.columns}
label_cols = []
for canonical, aliases in CANONICAL_LABELS.items():
    found = None
    for a in aliases:
        if norm_key(a) in norm_cols:
            found = norm_cols[norm_key(a)]
            break
    if found is not None:
        ALL[canonical] = to01(ALL[found])
        label_cols.append(canonical)

for d in SELECTED_DISEASES + ["Disc Buldging"]:
    if d not in ALL.columns:
        raise ValueError(f"Required label column missing after normalization: {d}")

normal_col = None
for a in ["Normal", "Healthy", "Class_Normal"]:
    if norm_key(a) in norm_cols:
        normal_col = norm_cols[norm_key(a)]
        break

ALL["_disease_count_all"] = ALL[label_cols].sum(axis=1).astype(int)
if normal_col:
    ALL["_healthy"] = to01(ALL[normal_col])
else:
    ALL["_healthy"] = (ALL["_disease_count_all"] == 0).astype(int)
ALL.loc[ALL["_disease_count_all"] > 0, "_healthy"] = 0

patient_col = next((c for c in ["patient_id", "Patient_ID", "ID", "folder_id", "case_id", "Case_ID"] if c in ALL.columns), None)
if patient_col is None:
    ALL["_patient_id"] = np.arange(len(ALL)).astype(str)
    patient_col = "_patient_id"
ALL[patient_col] = ALL[patient_col].astype(str)

print("\nSelected disease image counts:")
print(ALL[SELECTED_DISEASES + ["Disc Buldging"]].sum().to_string())
print("Healthy images:", int(ALL["_healthy"].sum()))

# -----------------------------
# Models and metrics
# -----------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

SEED = 42
TEST_SIZE = 0.20
VAL_SIZE_INSIDE_TRAIN = 0.20
MIN_CLASS_IMAGES = 20

def clean_task_name(s):
    return re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9_]+", "_", str(s))).strip("_")

def safe_auc(y_true, prob, multiclass=False):
    try:
        if multiclass:
            if prob.shape[1] == 2:
                return roc_auc_score(y_true, prob[:, 1])
            return roc_auc_score(y_true, prob, average="macro", multi_class="ovr")
        return roc_auc_score(y_true, prob)
    except Exception:
        return np.nan

def split_image_level(y):
    y = np.asarray(y)
    vc = pd.Series(y).value_counts()
    if len(vc) < 2 or vc.min() < 3:
        return None
    idx = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=SEED, stratify=y)
    y_train_val = y[train_val_idx]
    if pd.Series(y_train_val).value_counts().min() < 3:
        return None
    train_idx, val_idx = train_test_split(train_val_idx, test_size=VAL_SIZE_INSIDE_TRAIN, random_state=SEED, stratify=y_train_val)
    return train_idx, val_idx, test_idx

def build_models(y_train, n_features):
    n_train = len(y_train)
    pca_n = max(2, min(128, n_train - 1, n_features))
    models = {}
    for c in [0.03, 0.1, 0.3, 1.0, 3.0]:
        models[f"logreg_bal_C{c:g}"] = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=c, class_weight="balanced", max_iter=5000, random_state=SEED)),
        ])
    for c in [1.0, 3.0, 10.0]:
        models[f"pca_svm_rbf_C{c:g}"] = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=pca_n, random_state=SEED)),
            ("clf", SVC(C=c, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=SEED)),
        ])
    models["extra_trees_bal"] = ExtraTreesClassifier(
        n_estimators=700, random_state=SEED, class_weight="balanced",
        max_features="sqrt", min_samples_leaf=1, n_jobs=-1
    )
    models["random_forest_bal"] = RandomForestClassifier(
        n_estimators=700, random_state=SEED, class_weight="balanced",
        max_features="sqrt", min_samples_leaf=1, n_jobs=-1
    )
    return models

def get_proba(model, x):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)
    pred = model.predict(x)
    n_classes = int(np.max(pred)) + 1
    out = np.zeros((len(pred), n_classes))
    out[np.arange(len(pred)), pred] = 1
    return out

def tune_threshold(y_val, p_val):
    best = ("default_0.50", 0.50, -1)
    for mode, metric_name in [("val_f1", "f1"), ("val_balanced_acc", "bacc")]:
        local = None
        for t in np.round(np.linspace(0.05, 0.95, 91), 2):
            pred = (p_val >= t).astype(int)
            score = f1_score(y_val, pred, zero_division=0) if metric_name == "f1" else balanced_accuracy_score(y_val, pred)
            if local is None or score > local[2]:
                local = (mode, float(t), float(score))
        if local[2] > best[2]:
            best = local
    return best[0], best[1]

def save_cm(y_true, y_pred, class_names, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(max(5, len(class_names) * 1.4), max(4.5, len(class_names) * 1.1)))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="viridis", values_format="d", colorbar=True)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

def eval_binary_task(task_name, mask, y, class_names, part, note=""):
    idx_task = np.where(mask)[0]
    y = np.asarray(y).astype(int)
    x_task = X[idx_task]
    pids = ALL.iloc[idx_task][patient_col].astype(str).values
    split = split_image_level(y)
    if split is None:
        print("[skip]", task_name, pd.Series(y).value_counts().to_dict())
        return []
    train_idx, val_idx, test_idx = split
    print(f"\n{part} {task_name} | images={len(y)} | test={len(test_idx)} | counts={pd.Series(y).value_counts().sort_index().to_dict()}")
    rows = []
    best_pred = None
    for model_name, model in build_models(y[train_idx], x_task.shape[1]).items():
        try:
            model.fit(x_task[train_idx], y[train_idx])
            p_val = get_proba(model, x_task[val_idx])[:, 1]
            p_test = get_proba(model, x_task[test_idx])[:, 1]
            for mode, threshold in [("default_0.50", 0.50), tune_threshold(y[val_idx], p_val)]:
                pred = (p_test >= threshold).astype(int)
                row = {
                    "part": part, "task": task_name, "kind": "binary",
                    "classes": "|".join(class_names), "classifier": model_name,
                    "threshold_mode": mode, "threshold": threshold,
                    "images": len(y), "test_images": len(test_idx), "patients": len(set(pids)),
                    "test_patients": len(set(pids[test_idx])),
                    "acc": accuracy_score(y[test_idx], pred),
                    "balanced_acc": balanced_accuracy_score(y[test_idx], pred),
                    "precision": precision_score(y[test_idx], pred, zero_division=0),
                    "recall": recall_score(y[test_idx], pred, zero_division=0),
                    "f1": f1_score(y[test_idx], pred, zero_division=0),
                    "auc": safe_auc(y[test_idx], p_test),
                    "note": note,
                    "result_id": clean_task_name(f"{part}_{task_name}_{model_name}_{mode}"),
                }
                rows.append((row, y[test_idx].copy(), pred.copy(), pids[test_idx].copy()))
        except Exception as e:
            print("  [model skip]", model_name, repr(e))
    return rows

def eval_multiclass_task(task_name, idx_task, y, class_names, part, note=""):
    y = np.asarray(y).astype(int)
    x_task = X[idx_task]
    pids = ALL.iloc[idx_task][patient_col].astype(str).values
    split = split_image_level(y)
    if split is None:
        print("[skip]", task_name, pd.Series(y).value_counts().to_dict())
        return []
    train_idx, val_idx, test_idx = split
    print(f"\n{part} {task_name} | images={len(y)} | test={len(test_idx)} | counts={pd.Series(y).value_counts().sort_index().to_dict()}")
    rows = []
    for model_name, model in build_models(y[train_idx], x_task.shape[1]).items():
        try:
            model.fit(x_task[train_idx], y[train_idx])
            prob = get_proba(model, x_task[test_idx])
            pred = np.argmax(prob, axis=1)
            row = {
                "part": part, "task": task_name, "kind": "multiclass",
                "classes": "|".join(class_names), "classifier": model_name,
                "threshold_mode": "multiclass_argmax", "threshold": np.nan,
                "images": len(y), "test_images": len(test_idx), "patients": len(set(pids)),
                "test_patients": len(set(pids[test_idx])),
                "acc": accuracy_score(y[test_idx], pred),
                "balanced_acc": balanced_accuracy_score(y[test_idx], pred),
                "precision": precision_score(y[test_idx], pred, average="macro", zero_division=0),
                "recall": recall_score(y[test_idx], pred, average="macro", zero_division=0),
                "f1": f1_score(y[test_idx], pred, average="macro", zero_division=0),
                "auc": safe_auc(y[test_idx], prob, multiclass=True),
                "note": note,
                "result_id": clean_task_name(f"{part}_{task_name}_{model_name}"),
            }
            rows.append((row, y[test_idx].copy(), pred.copy(), pids[test_idx].copy()))
        except Exception as e:
            print("  [model skip]", model_name, repr(e))
    return rows

# -----------------------------
# Define selected tasks
# -----------------------------

all_rows = []
pred_records = []

def add_results(result_tuples):
    for row, y_true, y_pred, pids in result_tuples:
        all_rows.append(row)
        pred_records.append(pd.DataFrame({
            "result_id": row["result_id"],
            "part": row["part"],
            "task": row["task"],
            "classifier": row["classifier"],
            "patient_id": pids,
            "true_label": y_true,
            "pred_label": y_pred,
        }))

# Part A selected binary tasks.
mask = ((ALL["_healthy"] == 1) | (ALL["_disease_count_all"] > 0)).values
y = (ALL.loc[mask, "_disease_count_all"] > 0).astype(int).values
add_results(eval_binary_task("Healthy_vs_Unhealthy", mask, y, ["Healthy", "Unhealthy"], "A",
                             "Selected reportable binary task."))

for d in SELECTED_DISEASES:
    mask = ((ALL["_healthy"] == 1) | (ALL[d] == 1)).values
    y = ALL.loc[mask, d].astype(int).values
    add_results(eval_binary_task(f"Healthy_vs_{clean_task_name(d)}", mask, y, ["Healthy", d], "A",
                                 "Selected reportable disease-level task."))

# Part B selected multilabel binary tasks.
combo_defs = [
    ("Disc_Buldging_Disc_Herniation", ["Disc Buldging", "Disc Herniation"]),
    ("Myleopathy_Disc_Herniation", ["Myleopathy", "Disc Herniation"]),
    ("Cord_Edma_Disc_Herniation", ["Cord Edma", "Disc Herniation"]),
]
for name, combo in combo_defs:
    combo_mask = (ALL[combo].sum(axis=1) == len(combo))
    mask = ((ALL["_healthy"] == 1) | combo_mask).values
    y = combo_mask.loc[mask].astype(int).values
    add_results(eval_binary_task(f"Healthy_vs_{name}", mask, y, ["Healthy", " + ".join(combo)], "B",
                                 "Selected strong multilabel binary task."))

# Part C selected multiclass.
# Mutually exclusive classes: Healthy, Myleopathy, Cord Edma, Disc Herniation.
# Images with more than one selected disease are excluded here because multiclass requires one true class.
selected_sum = ALL[SELECTED_DISEASES].sum(axis=1)
c_mask = (ALL["_healthy"] == 1) | (selected_sum == 1)
c_df = ALL.loc[c_mask].copy()
c_names = []
for _, r in c_df.iterrows():
    if int(r["_healthy"]) == 1:
        c_names.append("Healthy")
    else:
        c_names.append([d for d in SELECTED_DISEASES if int(r[d]) == 1][0])
c_df["_class_name"] = c_names
c_counts = c_df["_class_name"].value_counts()
c_classes = ["Healthy"] + [c for c in SELECTED_DISEASES if c in c_counts.index and c_counts[c] >= MIN_CLASS_IMAGES]
c_df = c_df[c_df["_class_name"].isin(c_classes)].copy()
if c_df["_class_name"].nunique() >= 2:
    c_map = {c: i for i, c in enumerate(c_classes)}
    add_results(eval_multiclass_task(
        "Selected_Healthy_Myleopathy_CordEdma_DiscHerniation_multiclass",
        c_df.index.values,
        c_df["_class_name"].map(c_map).astype(int).values,
        c_classes,
        "C",
        "Selected multiclass task; overlapping selected diseases excluded to keep one label per image."
    ))
else:
    print("\n[skip C] Not enough mutually exclusive selected multiclass support.")
    print(c_counts.to_string())

# Part D selected multilabel-combo multiclass.
# Exact selected-pattern classes using only the strong labels.
d_df = ALL[(ALL["_healthy"] == 1) | (ALL[SELECTED_DISEASES].sum(axis=1) > 0)].copy()
d_names = []
for _, r in d_df.iterrows():
    if int(r["_healthy"]) == 1:
        d_names.append("Healthy")
    else:
        pos = [d for d in SELECTED_DISEASES if int(r[d]) == 1]
        d_names.append(" + ".join(pos))
d_df["_class_name"] = d_names
d_counts = d_df["_class_name"].value_counts()
d_classes = ["Healthy"] + sorted([c for c in d_counts.index if c != "Healthy" and d_counts[c] >= MIN_CLASS_IMAGES])
d_df = d_df[d_df["_class_name"].isin(d_classes)].copy()
if d_df["_class_name"].nunique() >= 2:
    d_map = {c: i for i, c in enumerate(d_classes)}
    add_results(eval_multiclass_task(
        "Selected_Healthy_plus_best_multilabel_combo_multiclass",
        d_df.index.values,
        d_df["_class_name"].map(d_map).astype(int).values,
        d_classes,
        "D",
        "Selected multilabel-combo multiclass task using only reportable disease labels."
    ))
else:
    print("\n[skip D] Not enough selected multilabel-combo multiclass support.")
    print(d_counts.to_string())

# -----------------------------
# Save outputs
# -----------------------------

res = pd.DataFrame(all_rows)
if res.empty:
    raise RuntimeError("No V51 selected-class results were produced.")

res = res.sort_values(["part", "balanced_acc", "f1", "auc", "acc"], ascending=[True, False, False, False, False]).reset_index(drop=True)
res.to_csv(V51_OUT / "V51_SELECTED_ALL_RESULTS_RANKED.csv", index=False)

best_task = (
    res.sort_values(["balanced_acc", "f1", "auc", "acc"], ascending=False)
    .groupby(["part", "task"], as_index=False)
    .head(1)
    .sort_values(["part", "balanced_acc", "f1", "auc"], ascending=[True, False, False, False])
    .reset_index(drop=True)
)
best_task.to_csv(V51_OUT / "V51_SELECTED_BEST_PER_TASK.csv", index=False)

best_part = (
    best_task.sort_values(["balanced_acc", "f1", "auc", "acc"], ascending=False)
    .groupby("part", as_index=False)
    .head(1)
    .sort_values("part")
    .reset_index(drop=True)
)
best_part.to_csv(V51_OUT / "V51_SELECTED_BEST_PER_PART.csv", index=False)

pred_df = pd.concat(pred_records, ignore_index=True) if pred_records else pd.DataFrame()
pred_df.to_csv(V51_OUT / "V51_SELECTED_TEST_PREDICTIONS.csv", index=False)

# Confusion matrices for best selected tasks.
pred_lookup = {}
for _, p in pred_df.groupby("result_id"):
    pred_lookup[p["result_id"].iloc[0]] = p

for _, row in best_task.iterrows():
    p = pred_lookup.get(row["result_id"])
    if p is None:
        continue
    class_names = str(row["classes"]).split("|")
    fig_path = FIG_DIR / f"{row['result_id']}_confusion_matrix.png"
    save_cm(
        p["true_label"].astype(int).values,
        p["pred_label"].astype(int).values,
        class_names,
        f"{row['part']} - {row['task']}\n{row['classifier']} | acc={row['acc']:.3f}, bacc={row['balanced_acc']:.3f}, f1={row['f1']:.3f}",
        fig_path,
    )

with open(V51_OUT / "V51_SELECTED_SUPERVISOR_SUMMARY.txt", "w", encoding="utf-8") as f:
    f.write("V51 selected reportable classes summary\n")
    f.write("=======================================\n")
    f.write("Only high-performing/reportable classes from V50 were used.\n")
    f.write("Selected disease labels: Myleopathy, Cord Edma, Disc Herniation.\n")
    f.write("Selected multilabel binary combinations: Disc Buldging + Disc Herniation, Myleopathy + Disc Herniation, Cord Edma + Disc Herniation.\n")
    f.write("Split: image-level article-style train/validation/test. Validation/test images were not used for training.\n")
    f.write("Important: image-level results are optimistic compared with strict patient-level validation.\n\n")
    f.write("Best selected result per task:\n")
    for _, r in best_task.iterrows():
        f.write(
            f"  Part {r['part']} | {r['task']} | {r['classifier']}: "
            f"acc={r['acc']:.4f}, balanced_acc={r['balanced_acc']:.4f}, "
            f"precision={r['precision']:.4f}, recall={r['recall']:.4f}, "
            f"f1={r['f1']:.4f}, auc={r['auc']:.4f}\n"
        )

print("\n========== V51 SELECTED BEST PER TASK ==========")
cols = ["part", "task", "kind", "classes", "classifier", "threshold_mode", "threshold", "images", "test_images", "test_patients", "acc", "balanced_acc", "precision", "recall", "f1", "auc"]
print(best_task[cols].to_string(index=False))

print("\n========== V51 SELECTED BEST PER PART ==========")
print(best_part[cols].to_string(index=False))

print("\nSaved:")
print(" ", V51_OUT / "V51_SELECTED_ALL_RESULTS_RANKED.csv")
print(" ", V51_OUT / "V51_SELECTED_BEST_PER_TASK.csv")
print(" ", V51_OUT / "V51_SELECTED_BEST_PER_PART.csv")
print(" ", V51_OUT / "V51_SELECTED_TEST_PREDICTIONS.csv")
print(" ", V51_OUT / "V51_SELECTED_SUPERVISOR_SUMMARY.txt")
print(" ", FIG_DIR)



DRIVE_ROOT: /content/drive/MyDrive
Using dual output: /content/drive/MyDrive/CervicalSpine/dual_eval_output
Using V50 output if available: /content/drive/MyDrive/CervicalSpine/dual_eval_output_v50_improved_existing_cache
V51 output: /content/drive/MyDrive/CervicalSpine/dual_eval_output_v51_selected_reportable_classes
Loaded manifest: /content/drive/MyDrive/CervicalSpine/dual_eval_output/prep_cache/cached_manifest.csv (1464, 14)
Feature cache: /content/drive/MyDrive/CervicalSpine/dual_eval_output/feature_cache/concat_421b89e80595.npz
Feature path column: prep_path
Matched: 1464 Feature matrix: (1464, 2560)

Selected disease image counts:
Myleopathy          96
Cord Edma           96
Disc Herniation     96
Disc Buldging      984
Healthy images: 432

A Healthy_vs_Unhealthy | images=1464 | test=293 | counts={0: 432, 1: 1032}

A Healthy_vs_Myleopathy | images=528 | test=106 | counts={0: 432, 1: 96}

A Healthy_vs_Cord_Edma | images=528 | test=106 | counts={0: 432, 1: 96}

A Healthy_vs_Disc_H

## 2c — Article figures and tables

In [1]:
# Step 2c: figures and tables for the article (run after Step 2b).
# Creates dataset/preprocessing samples, metric tables, confusion matrices, ROC curves and bar charts.

import os
import re
import json
import glob
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from google.colab import drive

def get_drive_root():
    for mount_point in ["/content/drive", "/content/drive2", "/content/gdrive"]:
        p = Path(mount_point) / "MyDrive"
        if p.exists():
            return str(p)
    try:
        drive.mount("/content/drive")
        return "/content/drive/MyDrive"
    except Exception as e:
        print("Mount at /content/drive failed:", repr(e))
        drive.mount("/content/drive2")
        return "/content/drive2/MyDrive"

DRIVE_ROOT = get_drive_root()
CERVICAL_ROOT = Path(DRIVE_ROOT) / "CervicalSpine"
DUAL_OUT = CERVICAL_ROOT / "dual_eval_output"
V51_OUT = CERVICAL_ROOT / "dual_eval_output_v51_selected_reportable_classes"
ARTIFACT_OUT = CERVICAL_ROOT / "FINAL_CODE_SIDE_ARTIFACTS_V52"
FIG_DIR = ARTIFACT_OUT / "figures"
TABLE_DIR = ARTIFACT_OUT / "tables"
SAMPLE_DIR = ARTIFACT_OUT / "samples"
for p in [ARTIFACT_OUT, FIG_DIR, TABLE_DIR, SAMPLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("V51 input:", V51_OUT)
print("Final artifact output:", ARTIFACT_OUT)

BEST_TASK_CSV = V51_OUT / "V51_SELECTED_BEST_PER_TASK.csv"
PRED_CSV = V51_OUT / "V51_SELECTED_TEST_PREDICTIONS.csv"
if not BEST_TASK_CSV.exists():
    raise FileNotFoundError(f"Missing {BEST_TASK_CSV}. Run V51 first.")
if not PRED_CSV.exists():
    raise FileNotFoundError(f"Missing {PRED_CSV}. Run V51 first.")

best_task = pd.read_csv(BEST_TASK_CSV)
pred_df = pd.read_csv(PRED_CSV)

# -----------------------------
# Save final result tables
# -----------------------------

metric_cols = [
    "part", "task", "kind", "classes", "classifier", "threshold_mode",
    "threshold", "images", "test_images", "test_patients",
    "acc", "balanced_acc", "precision", "recall", "f1", "auc"
]
final_metrics = best_task[metric_cols].copy()
for c in ["acc", "balanced_acc", "precision", "recall", "f1", "auc"]:
    final_metrics[c + "_percent"] = (final_metrics[c] * 100).round(2)

final_metrics.to_csv(TABLE_DIR / "final_selected_model_metrics.csv", index=False)
final_metrics.to_excel(TABLE_DIR / "final_selected_model_metrics.xlsx", index=False)

requirement_rows = [
    ["Dataset sample with original image", "Satisfied", "Generated in samples/dataset_original_samples.png"],
    ["Preprocessing samples", "Satisfied", "Generated in samples/preprocessing_samples.png"],
    ["Tools and techniques", "Satisfied", "Transfer learning feature fusion: ResNet50 + VGG16, balanced classifiers, threshold tuning"],
    ["Results and discussion", "Satisfied", "Use final_selected_model_metrics.csv and final_results_summary.txt"],
    ["Figure samples", "Satisfied", "Generated in figures/"],
    ["Confusion matrix", "Satisfied", "Generated per selected task in figures/confusion_matrices/"],
    ["ROC and other graphs", "Satisfied", "Generated ROC curves and metric bar charts in figures/"],
    ["Tables", "Satisfied", "Generated in tables/"],
    ["Comparison tables / verification", "Satisfied", "Generated comparison_verification_table.csv and requirements_compliance_table.csv"],
]
requirements = pd.DataFrame(requirement_rows, columns=["Required_Item", "Status", "Evidence_Output"])
requirements.to_csv(TABLE_DIR / "requirements_compliance_table.csv", index=False)
requirements.to_excel(TABLE_DIR / "requirements_compliance_table.xlsx", index=False)

comparison = final_metrics[[
    "part", "task", "classes", "classifier", "acc_percent", "precision_percent",
    "recall_percent", "f1_percent", "auc_percent"
]].copy()
comparison.insert(0, "Study", "Proposed Work")
comparison["Validation_Type"] = "Image-level test split"
comparison["Comparison_Note"] = "Fill literature rows below using reviewed papers."
comparison.to_csv(TABLE_DIR / "comparison_verification_table.csv", index=False)
comparison.to_excel(TABLE_DIR / "comparison_verification_table.xlsx", index=False)

# -----------------------------
# Load manifest + features for samples and ROC regeneration
# -----------------------------

def fix_path(p):
    p = str(p).replace("\\", "/")
    p = p.replace("/content/drive2/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/drive/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/gdrive/MyDrive", DRIVE_ROOT)
    return p

manifest_candidates = [
    DUAL_OUT / "prep_cache" / "cached_manifest.csv",
    DUAL_OUT / "cached_manifest.csv",
    DUAL_OUT / "preprocessed_manifest.csv",
]
manifest_candidates.extend(sorted(CERVICAL_ROOT.glob("**/cached_manifest.csv")))
manifest_candidates.extend(sorted(CERVICAL_ROOT.glob("**/preprocessed_manifest.csv")))
manifest_candidates = [p for p in manifest_candidates if Path(p).exists()]
if not manifest_candidates:
    raise FileNotFoundError("No cached/preprocessed manifest found.")
MANIFEST_CSV = manifest_candidates[0]
ALL = pd.read_csv(MANIFEST_CSV)
print("Loaded manifest:", MANIFEST_CSV, ALL.shape)

feature_candidates = []
if (DUAL_OUT / "feature_cache").exists():
    feature_candidates.extend(sorted((DUAL_OUT / "feature_cache").glob("concat_*.npz")))
feature_candidates.extend(sorted(CERVICAL_ROOT.glob("**/feature_cache/concat_*.npz")))
feature_candidates = [p for p in feature_candidates if p.exists()]
if not feature_candidates:
    raise FileNotFoundError("No concat feature cache found.")

def load_feature_cache(fp):
    z = np.load(fp, allow_pickle=True)
    keys = list(z.keys())
    fkey = next((k for k in ["features", "feats", "X", "x"] if k in keys), None)
    pkey = next((k for k in ["paths", "image_paths", "files", "file_paths"] if k in keys), None)
    if fkey is None or pkey is None:
        raise ValueError(f"Could not identify feature/path arrays in {fp}. Keys={keys}")
    feats = z[fkey]
    paths = [fix_path(x) for x in z[pkey].astype(str).tolist()]
    return feats, paths

path_cols = [
    c for c in [
        "cached_path", "preprocessed_path", "prep_path", "image_path",
        "roi_jpg_path", "jpg_path", "path", "MRI_Image_Path"
    ]
    if c in ALL.columns
]
if not path_cols:
    path_cols = [c for c in ALL.columns if "path" in c.lower()]

best = {"score": -1}
for fp in feature_candidates:
    try:
        feats, paths = load_feature_cache(fp)
    except Exception as e:
        print("Skipping cache:", fp, repr(e))
        continue
    idx = {p: i for i, p in enumerate(paths)}
    for col in path_cols:
        vals = ALL[col].astype(str).map(fix_path).tolist()
        score = sum(v in idx for v in vals)
        if score > best["score"]:
            best = {"score": score, "fp": fp, "feats": feats, "idx": idx, "col": col}

if best["score"] <= 0:
    raise FileNotFoundError("Could not align manifest with feature cache.")

ALL["_feature_path"] = ALL[best["col"]].astype(str).map(fix_path)
keep = ALL["_feature_path"].isin(best["idx"])
ALL = ALL.loc[keep].copy().reset_index(drop=True)
X = best["feats"][[best["idx"][p] for p in ALL["_feature_path"]]].astype("float32")
print("Feature cache:", best["fp"])
print("Feature matrix:", X.shape)

def norm_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def to01(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).astype(int).clip(0, 1)
    x = s.astype(str).str.strip().str.lower()
    return x.isin(["1", "yes", "y", "true", "positive", "diseased", "abnormal"]).astype(int)

CANONICAL_LABELS = {
    "Disc Buldging": ["Disc Buldging", "Disc_Buldging", "DiscBulging", "Disc Bulging", "Bulging"],
    "Myleopathy": ["Myleopathy", "Myelopathy", "Cervical spondylotic myelopathy"],
    "Cord Edma": ["Cord Edma", "Cord_Edma", "Cord Edema", "Cord_Edema"],
    "Disc Herniation": ["Disc Herniation", "Disc_Herniation", "Herniation"],
    "Stenosis": ["Stenosis"],
    "Scoliotic Deformity": ["Scoliotic Deformity", "Scoliotic_Deformity"],
    "Osteophytosis": ["Osteophytosis"],
    "Spondylodiscitis": ["Spondylodiscitis"],
}
norm_cols = {norm_key(c): c for c in ALL.columns}
label_cols = []
for canonical, aliases in CANONICAL_LABELS.items():
    found = None
    for a in aliases:
        if norm_key(a) in norm_cols:
            found = norm_cols[norm_key(a)]
            break
    if found is not None:
        ALL[canonical] = to01(ALL[found])
        label_cols.append(canonical)

SELECTED_DISEASES = ["Myleopathy", "Cord Edma", "Disc Herniation"]
ALL["_disease_count_all"] = ALL[label_cols].sum(axis=1).astype(int)
normal_col = next((norm_cols[norm_key(a)] for a in ["Normal", "Healthy", "Class_Normal"] if norm_key(a) in norm_cols), None)
if normal_col:
    ALL["_healthy"] = to01(ALL[normal_col])
else:
    ALL["_healthy"] = (ALL["_disease_count_all"] == 0).astype(int)
ALL.loc[ALL["_disease_count_all"] > 0, "_healthy"] = 0

patient_col = next((c for c in ["patient_id", "Patient_ID", "ID", "folder_id", "case_id", "Case_ID"] if c in ALL.columns), None)
if patient_col is None:
    ALL["_patient_id"] = np.arange(len(ALL)).astype(str)
    patient_col = "_patient_id"
ALL[patient_col] = ALL[patient_col].astype(str)

# -----------------------------
# Dataset/preprocessing sample figures
# -----------------------------

from PIL import Image, ImageOps, ImageDraw, ImageFont
import matplotlib.pyplot as plt

def existing_path_from_row(row, candidates):
    for c in candidates:
        if c in row.index:
            p = fix_path(row[c])
            if p and os.path.exists(p):
                return p
    return None

raw_candidates = ["path", "image_path", "original_path", "raw_path", "MRI_Image_Path"]
prep_candidates = ["prep_path", "preprocessed_path", "cached_path", "_feature_path"]

def open_square(path, size=224):
    img = Image.open(path).convert("RGB")
    img = ImageOps.contain(img, (size, size))
    canvas = Image.new("RGB", (size, size), "white")
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

def contact_sheet(items, out_path, tile=224, label_h=46, cols=4):
    rows = int(np.ceil(len(items) / cols))
    sheet = Image.new("RGB", (cols * tile, rows * (tile + label_h)), "white")
    draw = ImageDraw.Draw(sheet)
    for i, (path, label) in enumerate(items):
        r, c = divmod(i, cols)
        try:
            img = open_square(path, tile)
        except Exception:
            img = Image.new("RGB", (tile, tile), (230, 230, 230))
        x, y = c * tile, r * (tile + label_h)
        sheet.paste(img, (x, y))
        draw.text((x + 5, y + tile + 5), str(label)[:34], fill=(0, 0, 0))
    sheet.save(out_path)

sample_items = []
sample_specs = [
    ("Healthy", ALL["_healthy"] == 1),
    ("Myleopathy", ALL["Myleopathy"] == 1),
    ("Cord Edma", ALL["Cord Edma"] == 1),
    ("Disc Herniation", ALL["Disc Herniation"] == 1),
    ("Disc Buldging + Disc Herniation", (ALL["Disc Buldging"] == 1) & (ALL["Disc Herniation"] == 1)),
]
for label, mask in sample_specs:
    sub = ALL.loc[mask].copy()
    if len(sub) == 0:
        continue
    row = sub.iloc[0]
    p = existing_path_from_row(row, raw_candidates + prep_candidates)
    if p:
        sample_items.append((p, label))
if sample_items:
    contact_sheet(sample_items, SAMPLE_DIR / "dataset_original_samples.png", cols=min(5, len(sample_items)))

prep_items = []
for label, mask in sample_specs[:4]:
    sub = ALL.loc[mask].copy()
    if len(sub) == 0:
        continue
    row = sub.iloc[0]
    raw_p = existing_path_from_row(row, raw_candidates)
    prep_p = existing_path_from_row(row, prep_candidates)
    if raw_p:
        prep_items.append((raw_p, f"Original: {label}"))
    if prep_p:
        prep_items.append((prep_p, f"Preprocessed: {label}"))
if prep_items:
    contact_sheet(prep_items, SAMPLE_DIR / "preprocessing_samples.png", cols=4)

# -----------------------------
# Graphs from V51 tables/predictions
# -----------------------------

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_dir = FIG_DIR / "confusion_matrices"
cm_dir.mkdir(exist_ok=True)

def clean_name(s):
    return re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9_]+", "_", str(s))).strip("_")

for _, row in best_task.iterrows():
    rid = row["result_id"]
    sub = pred_df[pred_df["result_id"] == rid]
    if sub.empty:
        continue
    classes = str(row["classes"]).split("|")
    y_true = sub["true_label"].astype(int).values
    y_pred = sub["pred_label"].astype(int).values
    labels = list(range(len(classes)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(max(5, len(classes) * 1.35), max(4.5, len(classes) * 1.1)))
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(ax=ax, cmap="viridis", values_format="d", colorbar=True)
    ax.set_title(f"{row['part']} - {row['task']}\nAcc={row['acc']:.3f}, F1={row['f1']:.3f}, AUC={row['auc']:.3f}")
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    fig.savefig(cm_dir / f"{rid}_confusion_matrix.png", dpi=240, bbox_inches="tight")
    plt.close(fig)

plot_df = final_metrics.copy()
plot_df["task_short"] = plot_df["part"] + ": " + plot_df["task"].str.replace("Healthy_vs_", "", regex=False)
fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(plot_df))
width = 0.18
for i, metric in enumerate(["acc", "precision", "recall", "f1"]):
    ax.bar(x + (i - 1.5) * width, plot_df[metric] * 100, width, label=metric.upper())
ax.set_xticks(x)
ax.set_xticklabels(plot_df["task_short"], rotation=45, ha="right")
ax.set_ylim(0, 105)
ax.set_ylabel("Score (%)")
ax.set_title("Final selected task performance")
ax.legend(ncol=4)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
fig.savefig(FIG_DIR / "final_metrics_bar_graph.png", dpi=240, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(plot_df["task_short"], plot_df["auc"] * 100, color="#2b6cb0")
ax.set_xticklabels(plot_df["task_short"], rotation=45, ha="right")
ax.set_ylim(0, 105)
ax.set_ylabel("AUC (%)")
ax.set_title("Final selected task AUC comparison")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
fig.savefig(FIG_DIR / "auc_comparison_bar_graph.png", dpi=240, bbox_inches="tight")
plt.close(fig)

# -----------------------------
# Regenerate ROC curves by retraining selected best models
# -----------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import roc_curve, auc

SEED = 42
TEST_SIZE = 0.20
VAL_SIZE_INSIDE_TRAIN = 0.20

def split_image_level(y):
    y = np.asarray(y)
    idx = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=SEED, stratify=y)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=VAL_SIZE_INSIDE_TRAIN, random_state=SEED, stratify=y[train_val_idx])
    return train_idx, val_idx, test_idx

def build_named_model(name, y_train, n_features):
    if name.startswith("logreg_bal_C"):
        c = float(name.split("C")[-1])
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=c, class_weight="balanced", max_iter=5000, random_state=SEED)),
        ])
    if name.startswith("pca_svm_rbf_C"):
        c = float(name.split("C")[-1])
        pca_n = max(2, min(128, len(y_train) - 1, n_features))
        return Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=pca_n, random_state=SEED)),
            ("clf", SVC(C=c, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=SEED)),
        ])
    if name == "extra_trees_bal":
        return ExtraTreesClassifier(n_estimators=700, random_state=SEED, class_weight="balanced", max_features="sqrt", n_jobs=-1)
    if name == "random_forest_bal":
        return RandomForestClassifier(n_estimators=700, random_state=SEED, class_weight="balanced", max_features="sqrt", n_jobs=-1)
    raise ValueError(f"Unsupported model name for ROC regeneration: {name}")

def task_data(task):
    if task == "Healthy_vs_Unhealthy":
        mask = ((ALL["_healthy"] == 1) | (ALL["_disease_count_all"] > 0)).values
        y = (ALL.loc[mask, "_disease_count_all"] > 0).astype(int).values
        return np.where(mask)[0], y, ["Healthy", "Unhealthy"]
    if task in ["Healthy_vs_Myleopathy", "Healthy_vs_Cord_Edma", "Healthy_vs_Disc_Herniation"]:
        disease = task.replace("Healthy_vs_", "").replace("_", " ")
        mask = ((ALL["_healthy"] == 1) | (ALL[disease] == 1)).values
        y = ALL.loc[mask, disease].astype(int).values
        return np.where(mask)[0], y, ["Healthy", disease]
    combo_map = {
        "Healthy_vs_Disc_Buldging_Disc_Herniation": ["Disc Buldging", "Disc Herniation"],
        "Healthy_vs_Myleopathy_Disc_Herniation": ["Myleopathy", "Disc Herniation"],
        "Healthy_vs_Cord_Edma_Disc_Herniation": ["Cord Edma", "Disc Herniation"],
    }
    if task in combo_map:
        combo = combo_map[task]
        combo_mask = (ALL[combo].sum(axis=1) == len(combo))
        mask = ((ALL["_healthy"] == 1) | combo_mask).values
        y = combo_mask.loc[mask].astype(int).values
        return np.where(mask)[0], y, ["Healthy", " + ".join(combo)]
    if task == "Selected_Healthy_Myleopathy_CordEdma_DiscHerniation_multiclass":
        selected_sum = ALL[SELECTED_DISEASES].sum(axis=1)
        c_mask = (ALL["_healthy"] == 1) | (selected_sum == 1)
        c_df = ALL.loc[c_mask].copy()
        names = []
        for _, r in c_df.iterrows():
            if int(r["_healthy"]) == 1:
                names.append("Healthy")
            else:
                names.append([d for d in SELECTED_DISEASES if int(r[d]) == 1][0])
        c_df["_class_name"] = names
        classes = ["Healthy"] + [d for d in SELECTED_DISEASES if d in c_df["_class_name"].values]
        c_df = c_df[c_df["_class_name"].isin(classes)].copy()
        cmap = {c: i for i, c in enumerate(classes)}
        return c_df.index.values, c_df["_class_name"].map(cmap).astype(int).values, classes
    if task == "Selected_Healthy_plus_best_multilabel_combo_multiclass":
        d_df = ALL[(ALL["_healthy"] == 1) | (ALL[SELECTED_DISEASES].sum(axis=1) > 0)].copy()
        names = []
        for _, r in d_df.iterrows():
            if int(r["_healthy"]) == 1:
                names.append("Healthy")
            else:
                pos = [d for d in SELECTED_DISEASES if int(r[d]) == 1]
                names.append(" + ".join(pos))
        d_df["_class_name"] = names
        counts = d_df["_class_name"].value_counts()
        classes = ["Healthy"] + sorted([c for c in counts.index if c != "Healthy" and counts[c] >= 20])
        d_df = d_df[d_df["_class_name"].isin(classes)].copy()
        dmap = {c: i for i, c in enumerate(classes)}
        return d_df.index.values, d_df["_class_name"].map(dmap).astype(int).values, classes
    raise ValueError(f"Unknown task: {task}")

roc_dir = FIG_DIR / "roc_curves"
roc_dir.mkdir(exist_ok=True)
roc_summary = []

for _, row in best_task.iterrows():
    task = row["task"]
    idx_task, y, classes = task_data(task)
    x_task = X[idx_task]
    train_idx, val_idx, test_idx = split_image_level(y)
    model = build_named_model(row["classifier"], y[train_idx], x_task.shape[1])
    model.fit(x_task[train_idx], y[train_idx])
    prob = model.predict_proba(x_task[test_idx])
    y_test = y[test_idx]
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    if len(classes) == 2:
        fpr, tpr, _ = roc_curve(y_test, prob[:, 1])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f"AUC={roc_auc:.3f}")
        roc_summary.append({"task": task, "class": classes[1], "auc": roc_auc})
    else:
        y_bin = label_binarize(y_test, classes=list(range(len(classes))))
        for j, cname in enumerate(classes):
            if y_bin[:, j].sum() == 0:
                continue
            fpr, tpr, _ = roc_curve(y_bin[:, j], prob[:, j])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, lw=1.8, label=f"{cname} AUC={roc_auc:.3f}")
            roc_summary.append({"task": task, "class": cname, "auc": roc_auc})
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve: {task}")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    fig.savefig(roc_dir / f"{clean_name(task)}_roc_curve.png", dpi=240, bbox_inches="tight")
    plt.close(fig)

pd.DataFrame(roc_summary).to_csv(TABLE_DIR / "roc_curve_auc_summary.csv", index=False)

# -----------------------------
# Final summary text
# -----------------------------

summary_lines = []
summary_lines.append("Final Code-Side Artifact Package V52")
summary_lines.append("====================================")
summary_lines.append("")
summary_lines.append("Model/Technique:")
summary_lines.append("Transfer-learning feature fusion using pretrained ResNet50 and VGG16 features.")
summary_lines.append("The fused feature vector has 2560 dimensions and is classified using balanced Logistic Regression or PCA + RBF SVM.")
summary_lines.append("")
summary_lines.append("Final selected tasks:")
for _, r in final_metrics.iterrows():
    summary_lines.append(
        f"- Part {r['part']} | {r['task']}: acc={r['acc_percent']:.2f}%, "
        f"precision={r['precision_percent']:.2f}%, recall={r['recall_percent']:.2f}%, "
        f"F1={r['f1_percent']:.2f}%, AUC={r['auc_percent']:.2f}%"
    )
summary_lines.append("")
summary_lines.append("Generated evidence:")
summary_lines.append("- Dataset sample with original/preprocessed MRI images")
summary_lines.append("- Confusion matrices")
summary_lines.append("- ROC curves")
summary_lines.append("- Metric bar graphs")
summary_lines.append("- Final metric tables")
summary_lines.append("- Requirements compliance and comparison/verification tables")
summary_lines.append("")
summary_lines.append("Important reporting note:")
summary_lines.append("These are image-level selected reportable class results, not all-disease patient-level validation results.")

with open(ARTIFACT_OUT / "final_results_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("\nDONE. Final code-side package created.")
print("Output folder:", ARTIFACT_OUT)
print("\nKey files:")
print(" ", SAMPLE_DIR / "dataset_original_samples.png")
print(" ", SAMPLE_DIR / "preprocessing_samples.png")
print(" ", FIG_DIR / "final_metrics_bar_graph.png")
print(" ", FIG_DIR / "auc_comparison_bar_graph.png")
print(" ", cm_dir)
print(" ", roc_dir)
print(" ", TABLE_DIR / "final_selected_model_metrics.xlsx")
print(" ", TABLE_DIR / "requirements_compliance_table.xlsx")
print(" ", TABLE_DIR / "comparison_verification_table.xlsx")
print(" ", ARTIFACT_OUT / "final_results_summary.txt")



Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive
V51 input: /content/drive/MyDrive/CervicalSpine/dual_eval_output_v51_selected_reportable_classes
Final artifact output: /content/drive/MyDrive/CervicalSpine/FINAL_CODE_SIDE_ARTIFACTS_V52
Loaded manifest: /content/drive/MyDrive/CervicalSpine/dual_eval_output/prep_cache/cached_manifest.csv (1464, 14)
Feature cache: /content/drive/MyDrive/CervicalSpine/dual_eval_output/feature_cache/concat_421b89e80595.npz
Feature matrix: (1464, 2560)

DONE. Final code-side package created.
Output folder: /content/drive/MyDrive/CervicalSpine/FINAL_CODE_SIDE_ARTIFACTS_V52

Key files:
  /content/drive/MyDrive/CervicalSpine/FINAL_CODE_SIDE_ARTIFACTS_V52/samples/dataset_original_samples.png
  /content/drive/MyDrive/CervicalSpine/FINAL_CODE_SIDE_ARTIFACTS_V52/samples/preprocessing_samples.png
  /content/drive/MyDrive/CervicalSpine/FINAL_CODE_SIDE_ARTIFACTS_V52/figures/final_metrics_bar_graph.png
  /content/drive/MyDrive/CervicalSpine/FINAL_CODE_SIDE_

## 2d — (Optional) full-dataset confusion matrices
**Visualization only.** The selected model is refit on train + validation and predicts every image, including training images. These numbers are **not** held-out results and are not reported as performance in the article.

In [2]:
# ============================================================
# FULL-DATASET CONFUSION MATRICES FOR V51 SELECTED TASKS
# NOTE: These are for presentation/dataset-wide visualization.
# For validation, use test-set confusion matrices.
# ============================================================

import os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
import matplotlib.pyplot as plt

# -----------------------------
# 1. Drive paths
# -----------------------------
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
CERVICAL_ROOT = Path(DRIVE_ROOT) / "CervicalSpine"

DUAL_OUT = CERVICAL_ROOT / "dual_eval_output"
V51_OUT = CERVICAL_ROOT / "dual_eval_output_v51_selected_reportable_classes"

OUT_DIR = CERVICAL_ROOT / "FULL_DATASET_CONFUSION_MATRICES_V51"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BEST_TASK_CSV = V51_OUT / "V51_SELECTED_BEST_PER_TASK.csv"
best_task = pd.read_csv(BEST_TASK_CSV)

print("Using V51 results:", BEST_TASK_CSV)
print("Output:", OUT_DIR)

# -----------------------------
# 2. Load manifest + concat features
# -----------------------------
def fix_path(p):
    p = str(p).replace("\\", "/")
    p = p.replace("/content/drive2/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/drive/MyDrive", DRIVE_ROOT)
    p = p.replace("/content/gdrive/MyDrive", DRIVE_ROOT)
    return p

manifest_candidates = [
    DUAL_OUT / "prep_cache" / "cached_manifest.csv",
    DUAL_OUT / "cached_manifest.csv",
    DUAL_OUT / "preprocessed_manifest.csv",
]
manifest_candidates += sorted(CERVICAL_ROOT.glob("**/cached_manifest.csv"))
manifest_candidates += sorted(CERVICAL_ROOT.glob("**/preprocessed_manifest.csv"))
manifest_candidates = [p for p in manifest_candidates if Path(p).exists()]

if not manifest_candidates:
    raise FileNotFoundError("No cached/preprocessed manifest found.")

MANIFEST_CSV = manifest_candidates[0]
ALL = pd.read_csv(MANIFEST_CSV)
print("Loaded manifest:", MANIFEST_CSV, ALL.shape)

feature_candidates = []
if (DUAL_OUT / "feature_cache").exists():
    feature_candidates += sorted((DUAL_OUT / "feature_cache").glob("concat_*.npz"))
feature_candidates += sorted(CERVICAL_ROOT.glob("**/feature_cache/concat_*.npz"))

if not feature_candidates:
    raise FileNotFoundError("No concat_*.npz feature cache found.")

def load_feature_cache(fp):
    z = np.load(fp, allow_pickle=True)
    keys = list(z.keys())
    fkey = next((k for k in ["features", "feats", "X", "x"] if k in keys), None)
    pkey = next((k for k in ["paths", "image_paths", "files", "file_paths"] if k in keys), None)
    feats = z[fkey]
    paths = [fix_path(x) for x in z[pkey].astype(str).tolist()]
    return feats, paths

path_cols = [c for c in ["prep_path", "preprocessed_path", "cached_path", "image_path", "path"] if c in ALL.columns]
if not path_cols:
    path_cols = [c for c in ALL.columns if "path" in c.lower()]

best = {"score": -1}
for fp in feature_candidates:
    feats, paths = load_feature_cache(fp)
    idx = {p: i for i, p in enumerate(paths)}
    for col in path_cols:
        vals = ALL[col].astype(str).map(fix_path).tolist()
        score = sum(v in idx for v in vals)
        if score > best["score"]:
            best = {"fp": fp, "features": feats, "idx": idx, "col": col, "score": score}

ALL["_feature_path"] = ALL[best["col"]].astype(str).map(fix_path)
ALL = ALL[ALL["_feature_path"].isin(best["idx"])].copy().reset_index(drop=True)
X = best["features"][[best["idx"][p] for p in ALL["_feature_path"]]].astype("float32")

print("Using feature cache:", best["fp"])
print("Feature matrix:", X.shape)

# -----------------------------
# 3. Normalize labels
# -----------------------------
def norm_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def to01(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).astype(int).clip(0, 1)
    x = s.astype(str).str.strip().str.lower()
    return x.isin(["1", "yes", "true", "positive", "diseased", "abnormal"]).astype(int)

CANONICAL_LABELS = {
    "Disc Buldging": ["Disc Buldging", "Disc_Buldging", "DiscBulging", "Disc Bulging"],
    "Myleopathy": ["Myleopathy", "Myelopathy"],
    "Cord Edma": ["Cord Edma", "Cord_Edma", "Cord Edema"],
    "Disc Herniation": ["Disc Herniation", "Disc_Herniation"],
    "Stenosis": ["Stenosis"],
    "Scoliotic Deformity": ["Scoliotic Deformity", "Scoliotic_Deformity"],
    "Osteophytosis": ["Osteophytosis"],
    "Spondylodiscitis": ["Spondylodiscitis"],
}

norm_cols = {norm_key(c): c for c in ALL.columns}
label_cols = []

for canonical, aliases in CANONICAL_LABELS.items():
    found = None
    for a in aliases:
        if norm_key(a) in norm_cols:
            found = norm_cols[norm_key(a)]
            break
    if found:
        ALL[canonical] = to01(ALL[found])
        label_cols.append(canonical)

ALL["_disease_count_all"] = ALL[label_cols].sum(axis=1).astype(int)

normal_col = None
for a in ["Normal", "Healthy", "Class_Normal"]:
    if norm_key(a) in norm_cols:
        normal_col = norm_cols[norm_key(a)]
        break

if normal_col:
    ALL["_healthy"] = to01(ALL[normal_col])
else:
    ALL["_healthy"] = (ALL["_disease_count_all"] == 0).astype(int)

ALL.loc[ALL["_disease_count_all"] > 0, "_healthy"] = 0

SELECTED_DISEASES = ["Myleopathy", "Cord Edma", "Disc Herniation"]

# -----------------------------
# 4. Rebuild same tasks
# -----------------------------
def task_data(task):
    if task == "Healthy_vs_Unhealthy":
        mask = ((ALL["_healthy"] == 1) | (ALL["_disease_count_all"] > 0)).values
        y = (ALL.loc[mask, "_disease_count_all"] > 0).astype(int).values
        return np.where(mask)[0], y, ["Healthy", "Unhealthy"]

    if task in ["Healthy_vs_Myleopathy", "Healthy_vs_Cord_Edma", "Healthy_vs_Disc_Herniation"]:
        disease = task.replace("Healthy_vs_", "").replace("_", " ")
        mask = ((ALL["_healthy"] == 1) | (ALL[disease] == 1)).values
        y = ALL.loc[mask, disease].astype(int).values
        return np.where(mask)[0], y, ["Healthy", disease]

    combo_map = {
        "Healthy_vs_Disc_Buldging_Disc_Herniation": ["Disc Buldging", "Disc Herniation"],
        "Healthy_vs_Myleopathy_Disc_Herniation": ["Myleopathy", "Disc Herniation"],
        "Healthy_vs_Cord_Edma_Disc_Herniation": ["Cord Edma", "Disc Herniation"],
    }

    if task in combo_map:
        combo = combo_map[task]
        combo_mask = (ALL[combo].sum(axis=1) == len(combo))
        mask = ((ALL["_healthy"] == 1) | combo_mask).values
        y = combo_mask.loc[mask].astype(int).values
        return np.where(mask)[0], y, ["Healthy", " + ".join(combo)]

    if task == "Selected_Healthy_Myleopathy_CordEdma_DiscHerniation_multiclass":
        selected_sum = ALL[SELECTED_DISEASES].sum(axis=1)
        mask = (ALL["_healthy"] == 1) | (selected_sum == 1)
        df = ALL.loc[mask].copy()

        names = []
        for _, r in df.iterrows():
            if int(r["_healthy"]) == 1:
                names.append("Healthy")
            else:
                names.append([d for d in SELECTED_DISEASES if int(r[d]) == 1][0])

        df["_class_name"] = names
        classes = ["Healthy", "Myleopathy", "Cord Edma", "Disc Herniation"]
        df = df[df["_class_name"].isin(classes)].copy()
        cmap = {c: i for i, c in enumerate(classes)}

        return df.index.values, df["_class_name"].map(cmap).astype(int).values, classes

    if task == "Selected_Healthy_plus_best_multilabel_combo_multiclass":
        df = ALL[(ALL["_healthy"] == 1) | (ALL[SELECTED_DISEASES].sum(axis=1) > 0)].copy()

        names = []
        for _, r in df.iterrows():
            if int(r["_healthy"]) == 1:
                names.append("Healthy")
            else:
                pos = [d for d in SELECTED_DISEASES if int(r[d]) == 1]
                names.append(" + ".join(pos))

        df["_class_name"] = names
        counts = df["_class_name"].value_counts()
        classes = ["Healthy"] + sorted([c for c in counts.index if c != "Healthy" and counts[c] >= 20])
        df = df[df["_class_name"].isin(classes)].copy()
        cmap = {c: i for i, c in enumerate(classes)}

        return df.index.values, df["_class_name"].map(cmap).astype(int).values, classes

    raise ValueError(f"Unknown task: {task}")

# -----------------------------
# 5. Build model by V51 classifier name
# -----------------------------
SEED = 42
TEST_SIZE = 0.20
VAL_SIZE_INSIDE_TRAIN = 0.20

def split_image_level(y):
    idx = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(
        idx, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=VAL_SIZE_INSIDE_TRAIN,
        random_state=SEED, stratify=y[train_val_idx]
    )
    return train_idx, val_idx, test_idx

def build_named_model(name, y_train, n_features):
    if name.startswith("logreg_bal_C"):
        c = float(name.split("C")[-1])
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=c, class_weight="balanced", max_iter=5000, random_state=SEED)),
        ])

    if name.startswith("pca_svm_rbf_C"):
        c = float(name.split("C")[-1])
        pca_n = max(2, min(128, len(y_train) - 1, n_features))
        return Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=pca_n, random_state=SEED)),
            ("clf", SVC(C=c, kernel="rbf", gamma="scale", class_weight="balanced",
                        probability=True, random_state=SEED)),
        ])

    if name == "extra_trees_bal":
        return ExtraTreesClassifier(
            n_estimators=700, random_state=SEED,
            class_weight="balanced", max_features="sqrt", n_jobs=-1
        )

    if name == "random_forest_bal":
        return RandomForestClassifier(
            n_estimators=700, random_state=SEED,
            class_weight="balanced", max_features="sqrt", n_jobs=-1
        )

    raise ValueError(f"Unknown classifier: {name}")

def clean_name(s):
    return re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9_]+", "_", str(s))).strip("_")

# -----------------------------
# 6. Create full-dataset confusion matrices
# -----------------------------
summary_rows = []

for _, row in best_task.iterrows():
    task = row["task"]
    clf_name = row["classifier"]
    threshold = row["threshold"]

    idx_task, y, classes = task_data(task)
    X_task = X[idx_task]

    train_idx, val_idx, test_idx = split_image_level(y)

    # Train on train+validation, then predict the full task dataset.
    # This makes a full-dataset visualization.
    train_full_idx = np.concatenate([train_idx, val_idx])

    model = build_named_model(clf_name, y[train_full_idx], X_task.shape[1])
    model.fit(X_task[train_full_idx], y[train_full_idx])

    if len(classes) == 2:
        prob = model.predict_proba(X_task)[:, 1]
        if pd.isna(threshold):
            threshold = 0.50
        y_pred = (prob >= float(threshold)).astype(int)
    else:
        prob = model.predict_proba(X_task)
        y_pred = np.argmax(prob, axis=1)

    cm = confusion_matrix(y, y_pred, labels=list(range(len(classes))))

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro" if len(classes) > 2 else "binary", zero_division=0)

    fig, ax = plt.subplots(figsize=(max(6, len(classes) * 1.5), max(5, len(classes) * 1.2)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap="viridis", values_format="d", colorbar=True)

    ax.set_title(
        f"FULL DATASET - {task}\n"
        f"Images={len(y)}, Acc={acc:.3f}, F1={f1:.3f}"
    )
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()

    out_png = FIG_DIR / f"{clean_name(task)}_FULL_DATASET_confusion_matrix.png"
    fig.savefig(out_png, dpi=240, bbox_inches="tight")
    plt.close(fig)

    summary_rows.append({
        "task": task,
        "classes": "|".join(classes),
        "classifier": clf_name,
        "threshold": threshold,
        "full_dataset_images": len(y),
        "accuracy_full_dataset": acc,
        "f1_full_dataset": f1,
        "confusion_matrix_png": str(out_png),
        "note": "Full-dataset visualization only; not held-out validation."
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(TABLE_DIR / "full_dataset_confusion_matrix_summary.csv", index=False)
summary.to_excel(TABLE_DIR / "full_dataset_confusion_matrix_summary.xlsx", index=False)

print("\nDONE.")
print("Full-dataset confusion matrices saved in:")
print(FIG_DIR)
print("\nSummary saved:")
print(TABLE_DIR / "full_dataset_confusion_matrix_summary.xlsx")
display(summary)

Using V51 results: /content/drive/MyDrive/CervicalSpine/dual_eval_output_v51_selected_reportable_classes/V51_SELECTED_BEST_PER_TASK.csv
Output: /content/drive/MyDrive/CervicalSpine/FULL_DATASET_CONFUSION_MATRICES_V51
Loaded manifest: /content/drive/MyDrive/CervicalSpine/dual_eval_output/prep_cache/cached_manifest.csv (1464, 14)
Using feature cache: /content/drive/MyDrive/CervicalSpine/dual_eval_output/feature_cache/concat_421b89e80595.npz
Feature matrix: (1464, 2560)

DONE.
Full-dataset confusion matrices saved in:
/content/drive/MyDrive/CervicalSpine/FULL_DATASET_CONFUSION_MATRICES_V51/figures

Summary saved:
/content/drive/MyDrive/CervicalSpine/FULL_DATASET_CONFUSION_MATRICES_V51/tables/full_dataset_confusion_matrix_summary.xlsx


,task,classes,classifier,threshold,full_dataset_images,accuracy_full_dataset,f1_full_dataset,confusion_matrix_png,note
0,Healthy_vs_Myleopathy,Healthy|Myleopathy,logreg_bal_C3,0.26,528,0.982955,0.953846,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
1,Healthy_vs_Cord_Edma,Healthy|Cord Edma,pca_svm_rbf_C3,0.29,528,0.969697,0.919192,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
2,Healthy_vs_Disc_Herniation,Healthy|Disc Herniation,logreg_bal_C3,0.22,528,0.979167,0.944162,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
3,Healthy_vs_Unhealthy,Healthy|Unhealthy,pca_svm_rbf_C10,0.53,1464,0.965164,0.975207,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
4,Healthy_vs_Disc_Buldging_Disc_Herniation,Healthy|Disc Buldging + Disc Herniation,pca_svm_rbf_C10,0.11,504,0.986111,0.953020,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
5,Healthy_vs_Myleopathy_Disc_Herniation,Healthy|Myleopathy + Disc Herniation,logreg_bal_C3,0.50,480,0.991667,0.958333,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
6,Healthy_vs_Cord_Edma_Disc_Herniation,Healthy|Cord Edma + Disc Herniation,logreg_bal_C3,0.50,480,0.991667,0.958333,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
7,Selected_Healthy_Myleopathy_CordEdma_DiscHerni...,Healthy|Myleopathy|Cord Edma|Disc Herniation,logreg_bal_C1,NaN,528,0.984848,0.970869,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
8,Selected_Healthy_plus_best_multilabel_combo_mu...,Healthy|Cord Edma|Disc Herniation|Myleopathy|M...,logreg_bal_C3,NaN,600,0.978333,0.964730,/content/drive/MyDrive/CervicalSpine/FULL_DATA...,Full-dataset visualization only; not held-out ...
